# 11.6 重采样及频率转换

In [3]:
import pandas as pd
import numpy as np
dates = pd.date_range('2000-01-01', periods=100)
ts = pd.Series(np.random.randn(100), index=dates)
ts

2000-01-01   -0.685830
2000-01-02   -0.639954
2000-01-03   -1.285111
2000-01-04   -0.027319
2000-01-05   -0.107835
                ...   
2000-04-05    0.214310
2000-04-06    0.494675
2000-04-07   -0.174185
2000-04-08   -1.126423
2000-04-09    2.409967
Freq: D, Length: 100, dtype: float64

In [4]:
ts.resample('ME').mean()

2000-01-31    0.073730
2000-02-29   -0.107749
2000-03-31    0.289206
2000-04-30    0.860303
Freq: ME, dtype: float64

In [5]:
ts.resample('ME').mean().to_period('M')

2000-01    0.073730
2000-02   -0.107749
2000-03    0.289206
2000-04    0.860303
Freq: M, dtype: float64

## 11.6.1 降采样

In [6]:
dates = pd.date_range('2000-01-01', periods=12, freq='min')
ts = pd.Series(np.arange(12), index=dates)
ts

2000-01-01 00:00:00     0
2000-01-01 00:01:00     1
2000-01-01 00:02:00     2
2000-01-01 00:03:00     3
2000-01-01 00:04:00     4
2000-01-01 00:05:00     5
2000-01-01 00:06:00     6
2000-01-01 00:07:00     7
2000-01-01 00:08:00     8
2000-01-01 00:09:00     9
2000-01-01 00:10:00    10
2000-01-01 00:11:00    11
Freq: min, dtype: int64

In [7]:
ts.resample('5min').sum()  # 默认从左重采样

2000-01-01 00:00:00    10
2000-01-01 00:05:00    35
2000-01-01 00:10:00    21
Freq: 5min, dtype: int64

In [8]:
ts.resample('5min', closed='right').sum()  # 从右重采样

1999-12-31 23:55:00     0
2000-01-01 00:00:00    15
2000-01-01 00:05:00    40
2000-01-01 00:10:00    11
Freq: 5min, dtype: int64

In [9]:
# 上面分箱都是以左边界标记 传入 label='right' 就会以右边界标记
ts.resample('5min', closed='right', label='right').sum()

2000-01-01 00:00:00     0
2000-01-01 00:05:00    15
2000-01-01 00:10:00    40
2000-01-01 00:15:00    11
Freq: 5min, dtype: int64

In [10]:
# 做位移以分清时间戳代表的区间
from pandas.tseries.frequencies import to_offset
temp = ts.resample('5min', closed='right', label='right').sum()
temp.index = temp.index + to_offset('-1min')
temp

1999-12-31 23:59:00     0
2000-01-01 00:04:00    15
2000-01-01 00:09:00    40
2000-01-01 00:14:00    11
Freq: 5min, dtype: int64

#### 开-高-低-收(OHLC)重采样

In [11]:
ts.resample('5min').ohlc()

,open,high,low,close
2000-01-01 00:00:00,0,4,0,4
2000-01-01 00:05:00,5,9,5,9
2000-01-01 00:10:00,10,11,10,11


## 11.6.2 升采样和插值

In [2]:
import pandas as pd
import numpy as np
frame = pd.DataFrame(np.random.randn(2,4),index=pd.date_range('2000-01-01', periods=2, freq='W-WED'),columns=['Colorado','Texas','New York','Ohio'])
frame

,Colorado,Texas,New York,Ohio
2000-01-05,0.047066,-0.446453,0.887662,0.238204
2000-01-12,-0.902329,1.047286,2.034095,-0.531961


In [4]:
df_daily = frame.resample('D').asfreq()
df_daily

,Colorado,Texas,New York,Ohio
2000-01-05,0.047066,-0.446453,0.887662,0.238204
2000-01-06,NaN,NaN,NaN,NaN
2000-01-07,NaN,NaN,NaN,NaN
2000-01-08,NaN,NaN,NaN,NaN
2000-01-09,NaN,NaN,NaN,NaN
2000-01-10,NaN,NaN,NaN,NaN
2000-01-11,NaN,NaN,NaN,NaN
2000-01-12,-0.902329,1.047286,2.034095,-0.531961


In [5]:
frame.resample('D').ffill(limit=2)

,Colorado,Texas,New York,Ohio
2000-01-05,0.047066,-0.446453,0.887662,0.238204
2000-01-06,0.047066,-0.446453,0.887662,0.238204
2000-01-07,0.047066,-0.446453,0.887662,0.238204
2000-01-08,NaN,NaN,NaN,NaN
2000-01-09,NaN,NaN,NaN,NaN
2000-01-10,NaN,NaN,NaN,NaN
2000-01-11,NaN,NaN,NaN,NaN
2000-01-12,-0.902329,1.047286,2.034095,-0.531961


## 11.6.3 使用周期进行重采样

In [6]:
frame = pd.DataFrame(np.random.randn(24,4),index=pd.period_range('1-2000','12-2001',freq='M'),columns=['Colorado','Texas','New York','Ohio'])
frame.head()

,Colorado,Texas,New York,Ohio
2000-01,0.232155,1.102272,1.359806,-0.344050
2000-02,-2.144670,0.222804,-0.865327,-0.283989
2000-03,1.253593,-1.714858,-0.063498,0.289878
2000-04,-1.140581,0.227898,2.354394,-0.902441
2000-05,-1.645083,-1.221275,-0.001481,0.086134


In [9]:
annual_frame = frame.resample('Y-DEC').mean()
annual_frame

,Colorado,Texas,New York,Ohio
2000,0.091540,-0.623107,0.152762,-0.334025
2001,0.247956,0.507779,-0.072044,0.129736


In [11]:
# 默认放开头,即 convention='start'
annual_frame.resample('Q-DEC').ffill()

,Colorado,Texas,New York,Ohio
2000Q1,0.091540,-0.623107,0.152762,-0.334025
2000Q2,0.091540,-0.623107,0.152762,-0.334025
2000Q3,0.091540,-0.623107,0.152762,-0.334025
2000Q4,0.091540,-0.623107,0.152762,-0.334025
2001Q1,0.247956,0.507779,-0.072044,0.129736
2001Q2,0.247956,0.507779,-0.072044,0.129736
2001Q3,0.247956,0.507779,-0.072044,0.129736
2001Q4,0.247956,0.507779,-0.072044,0.129736


In [12]:
annual_frame.resample('Q-DEC', convention='end').asfreq()

,Colorado,Texas,New York,Ohio
2000Q4,0.091540,-0.623107,0.152762,-0.334025
2001Q1,NaN,NaN,NaN,NaN
2001Q2,NaN,NaN,NaN,NaN
2001Q3,NaN,NaN,NaN,NaN
2001Q4,0.247956,0.507779,-0.072044,0.129736


In [ ]:
# 周期的降采样目标频率必须是频率源的子周期 升采样必须是父周期

## 11.6.4 对分组时间进行重采样

In [13]:
N=15
times = pd.date_range('2017-5-20 00:00',freq='1min',periods=N)
df = pd.DataFrame({'time':times,'value':np.arange(N)})
df

,time,value
0,2017-05-20 00:00:00,0
1,2017-05-20 00:01:00,1
2,2017-05-20 00:02:00,2
3,2017-05-20 00:03:00,3
4,2017-05-20 00:04:00,4
5,2017-05-20 00:05:00,5
6,2017-05-20 00:06:00,6
7,2017-05-20 00:07:00,7
8,2017-05-20 00:08:00,8
9,2017-05-20 00:09:00,9


In [14]:
df.set_index('time').resample('5min').count()  # 注意set_index一步不可省略

,value
time,
2017-05-20 00:00:00,5
2017-05-20 00:05:00,5
2017-05-20 00:10:00,5


In [16]:
df2 = pd.DataFrame({'time':times.repeat(3),
                    'key':np.tile(['a','b','c'], N),  # 重复数组
                    'value':np.arange(N*3.)
                    })
df2.head(7)

,time,key,value
0,2017-05-20 00:00:00,a,0.0
1,2017-05-20 00:00:00,b,1.0
2,2017-05-20 00:00:00,c,2.0
3,2017-05-20 00:01:00,a,3.0
4,2017-05-20 00:01:00,b,4.0
5,2017-05-20 00:01:00,c,5.0
6,2017-05-20 00:02:00,a,6.0


In [17]:
time_key = pd.Grouper(freq='5min')
resampled = (df2.set_index('time').groupby(['key', time_key]).sum())
resampled

value
key time                      
a   2017-05-20 00:00:00   30.0
    2017-05-20 00:05:00  105.0
    2017-05-20 00:10:00  180.0
b   2017-05-20 00:00:00   35.0
    2017-05-20 00:05:00  110.0
    2017-05-20 00:10:00  185.0
c   2017-05-20 00:00:00   40.0
    2017-05-20 00:05:00  115.0
    2017-05-20 00:10:00  190.0

In [18]:
resampled.reset_index()

,key,time,value
0,a,2017-05-20 00:00:00,30.0
1,a,2017-05-20 00:05:00,105.0
2,a,2017-05-20 00:10:00,180.0
3,b,2017-05-20 00:00:00,35.0
4,b,2017-05-20 00:05:00,110.0
5,b,2017-05-20 00:10:00,185.0
6,c,2017-05-20 00:00:00,40.0
7,c,2017-05-20 00:05:00,115.0
8,c,2017-05-20 00:10:00,190.0


In [ ]:
# pd.Grouper必须使用时间作为pandas对象的索引

# End